# Interactive dataset filter and plots

Carica un CSV, filtra le colonne richieste e aggiorna i grafici con `ipywidgets`.

In [ ]:
from pathlib import Path
import re

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

FEATURES = ["gyration_radius", "flatness", "roughness", "scalar_prod"]
PLOT_MODES = [
    ("raw", None, "Raw RDF"),
    ("max_norm", "max", "Max-normalized RDF"),
    ("minmax_norm", "minmax", "Min-max normalized RDF"),
]
STATE = {
    "df": None,
    "ring_ids": [],
    "summary_types": [],
    "plot_limits": {},
    "bins": 30,
    "summary_path": None,
}

csv_path = widgets.Text(
    value="output_files/default/default_summary.csv",
    description="CSV",
    placeholder="Percorso o nome del file CSV",
    layout=widgets.Layout(width="70%"),
)
load_button = widgets.Button(description="Carica DataFrame", button_style="primary", icon="upload")
status = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="0.5rem"))

slider_style = {"description_width": "140px"}
slider_layout = widgets.Layout(width="95%")
sliders = {
    feature: widgets.FloatRangeSlider(
        description=feature,
        min=0.0,
        max=1.0,
        value=(0.0, 1.0),
        step=0.01,
        continuous_update=False,
        readout_format=".3f",
        disabled=True,
        style=slider_style,
        layout=slider_layout,
    )
    for feature in FEATURES
}


def _ordered_summary_types(values):
    preferred_order = ["weighted", "normal"]
    summary_types = [summary_type for summary_type in preferred_order if summary_type in values]
    for summary_type in values:
        if summary_type not in summary_types:
            summary_types.append(summary_type)
    return summary_types



def _ring_ids_from_columns(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}(\d+)_(?:mean|uncertainty)$")
    ring_ids = set()
    for column in columns:
        match = pattern.match(column)
        if match is not None:
            ring_ids.add(int(match.group(1)))
    return sorted(ring_ids)



def _finite_series(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return values[np.isfinite(values)]



def _safe_set_slider_range(slider, lower, upper):
    current_min = float(slider.min)
    current_max = float(slider.max)
    if lower > current_max:
        slider.max = upper
        slider.min = lower
    elif upper < current_min:
        slider.min = lower
        slider.max = upper
    else:
        slider.max = upper
        slider.min = lower
    slider.value = (lower, upper)
    slider.disabled = False



def _configure_slider(slider, series):
    values = _finite_series(series)
    if values.empty:
        slider.disabled = True
        slider.min = 0.0
        slider.max = 1.0
        slider.value = (0.0, 1.0)
        return None

    lower = float(values.min())
    upper = float(values.max())
    if np.isclose(lower, upper):
        padding = max(abs(lower) * 0.05, 1.0)
        lower -= padding
        upper += padding

    _safe_set_slider_range(slider, lower, upper)
    return lower, upper



def _mean_and_sample_uncertainty(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return float("nan"), float("nan")
    mean_value = float(np.mean(values))
    if len(values) == 1:
        return mean_value, 0.0
    uncertainty = float(np.std(values, ddof=1) / np.sqrt(len(values)))
    return mean_value, uncertainty



def _propagated_mean_uncertainty(uncertainties):
    uncertainties = np.asarray(uncertainties, dtype=float)
    uncertainties = uncertainties[np.isfinite(uncertainties)]
    if len(uncertainties) == 0:
        return float("nan")
    if len(uncertainties) == 1:
        return float(uncertainties[0])
    return float(np.sqrt(np.sum(uncertainties ** 2)) / len(uncertainties))



def _combined_uncertainty(sample_uncertainty, propagated_uncertainty):
    if not np.isfinite(sample_uncertainty) and not np.isfinite(propagated_uncertainty):
        return float("nan")
    if not np.isfinite(sample_uncertainty):
        return float(propagated_uncertainty)
    if not np.isfinite(propagated_uncertainty):
        return float(sample_uncertainty)
    return float(np.sqrt(sample_uncertainty ** 2 + propagated_uncertainty ** 2))



def _normalize_complex_series(values, uncertainties, mode):
    values = np.asarray(values, dtype=float)
    uncertainties = np.asarray(uncertainties, dtype=float)
    normalized_values = values.copy()
    normalized_uncertainties = uncertainties.copy()

    finite_values = values[np.isfinite(values)]
    if len(finite_values) == 0 or mode is None:
        return normalized_values, normalized_uncertainties

    if mode == "max":
        scale = float(np.max(finite_values))
        if scale <= 0:
            normalized_values[np.isfinite(normalized_values)] = 0.0
            normalized_uncertainties[np.isfinite(normalized_uncertainties)] = 0.0
            return normalized_values, normalized_uncertainties
        normalized_values = normalized_values / scale
        normalized_uncertainties = normalized_uncertainties / scale
        return normalized_values, normalized_uncertainties

    if mode == "minmax":
        min_value = float(np.min(finite_values))
        max_value = float(np.max(finite_values))
        scale = max_value - min_value
        if scale <= 0:
            normalized_values[np.isfinite(normalized_values)] = 0.0
            normalized_uncertainties[np.isfinite(normalized_uncertainties)] = 0.0
            return normalized_values, normalized_uncertainties
        normalized_values = (normalized_values - min_value) / scale
        normalized_uncertainties = normalized_uncertainties / scale
        return normalized_values, normalized_uncertainties

    raise ValueError(f"Unsupported normalization mode: {mode}")



def _aggregate_metric_by_summary_type(df_summary, summary_types, ring_ids, metric_prefix, normalization_mode=None):
    aggregated = {}
    for summary_type in summary_types:
        group = df_summary[df_summary["summary_type"] == summary_type]
        ring_values = {}
        per_ring_values = {rid: [] for rid in ring_ids}
        per_ring_uncertainties = {rid: [] for rid in ring_ids}

        if "complex_name" not in group.columns:
            raise ValueError("complex_name column is required to aggregate by complex")

        for _, complex_group in group.groupby("complex_name", sort=False):
            complex_means = []
            complex_uncertainties = []

            for rid in ring_ids:
                mean_col = f"{metric_prefix}_ring{rid}_mean"
                unc_col = f"{metric_prefix}_ring{rid}_uncertainty"
                if mean_col not in complex_group.columns:
                    complex_means.append(float("nan"))
                    complex_uncertainties.append(float("nan"))
                    continue

                value = float(complex_group[mean_col].iloc[0])
                uncertainty = float(complex_group[unc_col].iloc[0]) if unc_col in complex_group.columns else float("nan")
                complex_means.append(value)
                complex_uncertainties.append(uncertainty)

            normalized_means, normalized_uncertainties = _normalize_complex_series(
                np.array(complex_means, dtype=float),
                np.array(complex_uncertainties, dtype=float),
                normalization_mode,
            )

            for idx, rid in enumerate(ring_ids):
                if np.isfinite(normalized_means[idx]):
                    per_ring_values[rid].append(normalized_means[idx])
                if np.isfinite(normalized_uncertainties[idx]):
                    per_ring_uncertainties[rid].append(normalized_uncertainties[idx])

        for rid in ring_ids:
            finite_means = np.array(per_ring_values[rid], dtype=float)
            finite_uncertainties = np.array(per_ring_uncertainties[rid], dtype=float)
            if len(finite_means) == 0:
                ring_values[rid] = {
                    "mean": float("nan"),
                    "sample_uncertainty": float("nan"),
                    "propagated_uncertainty": float("nan"),
                    "combined_uncertainty": float("nan"),
                    "count": 0,
                }
                continue

            mean_value, sample_uncertainty = _mean_and_sample_uncertainty(finite_means)
            propagated_uncertainty = _propagated_mean_uncertainty(finite_uncertainties)
            combined_uncertainty = _combined_uncertainty(sample_uncertainty, propagated_uncertainty)
            ring_values[rid] = {
                "mean": mean_value,
                "sample_uncertainty": sample_uncertainty,
                "propagated_uncertainty": propagated_uncertainty,
                "combined_uncertainty": combined_uncertainty,
                "count": int(len(finite_means)),
            }
        aggregated[summary_type] = ring_values
    return aggregated



def _compute_panel_limits(metric_data, ring_ids, summary_types):
    finite_lower = []
    finite_upper = []
    for summary_type in summary_types:
        for rid in ring_ids:
            item = metric_data[summary_type][rid]
            mean_value = item["mean"]
            combined_uncertainty = item["combined_uncertainty"]
            if np.isfinite(mean_value):
                finite_lower.append(mean_value)
                finite_upper.append(mean_value)
                if np.isfinite(combined_uncertainty):
                    finite_lower.append(mean_value - 3.0 * combined_uncertainty)
                    finite_upper.append(mean_value + 3.0 * combined_uncertainty)

    if not finite_lower or not finite_upper:
        return (0.0, 1.0)

    lower = float(min(finite_lower))
    upper = float(max(finite_upper))
    if np.isclose(lower, upper):
        padding = max(abs(lower) * 0.05, 1.0)
        lower -= padding
        upper += padding
    else:
        padding = max((upper - lower) * 0.08, 1e-6)
        lower -= padding
        upper += padding

    return lower, upper



def _plot_metric_panel(ax, metric_data, ring_ids, summary_types, colors, markers, metric_label, value_label, y_limits, show_legend=False):
    x_positions = np.arange(len(ring_ids))
    offset_step = 0.18 / max(len(summary_types), 1)
    uncertainty_scale = 3.0

    for type_idx, summary_type in enumerate(summary_types):
        ring_values = metric_data[summary_type]
        means = np.array([ring_values[rid]["mean"] for rid in ring_ids], dtype=float)
        combined_uncertainties = np.array([ring_values[rid]["combined_uncertainty"] for rid in ring_ids], dtype=float)

        x_offset = (type_idx - len(summary_types) / 2 + 0.5) * offset_step
        x_pos = x_positions + x_offset
        color = colors.get(summary_type, "#444444")
        ax.errorbar(
            x_pos,
            means,
            yerr=combined_uncertainties * uncertainty_scale,
            fmt="none",
            ecolor=color,
            elinewidth=1.8,
            capsize=4,
            capthick=1.4,
            zorder=2,
        )
        ax.scatter(
            x_pos,
            means,
            s=18,
            marker=markers.get(summary_type, "o"),
            color=color,
            edgecolors="black",
            linewidth=0.8,
            alpha=0.95,
            zorder=3,
        )

    ax.set_xlabel("Ring ID", fontsize=12, fontweight="bold")
    ax.set_ylabel(value_label, fontsize=12, fontweight="bold")
    ax.set_title(metric_label, fontsize=13, fontweight="bold")
    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"{r}" for r in ring_ids])
    ax.set_xlim(-0.5, len(ring_ids) - 0.5)
    ax.set_ylim(y_limits)
    ax.set_autoscale_on(False)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    if show_legend:
        summary_handles = [
            Line2D(
                [0], [0],
                marker=markers.get(summary_type, "o"),
                linestyle="none",
                markerfacecolor=colors.get(summary_type, "#444444"),
                markeredgecolor="black",
                markersize=8,
                label=summary_type,
            )
            for summary_type in summary_types
        ]
        legend1 = ax.legend(handles=summary_handles, fontsize=10, loc="best")
        ax.add_artist(legend1)



def _prepare_state(df):
    ring_ids = _ring_ids_from_columns(df.columns, "physical_ring")
    if not ring_ids:
        raise ValueError("Nessuna colonna physical_ringN_mean/uncertainty trovata nel CSV")

    if "summary_type" not in df.columns:
        raise ValueError("summary_type column is required")
    if "complex_name" not in df.columns:
        raise ValueError("complex_name column is required")

    summary_types = _ordered_summary_types(df["summary_type"].dropna().unique().tolist())
    plot_limits = {}
    for mode_name, normalization_mode, _ in PLOT_MODES:
        physical_data = _aggregate_metric_by_summary_type(df, summary_types, ring_ids, "physical", normalization_mode=normalization_mode)
        zernike_data = _aggregate_metric_by_summary_type(df, summary_types, ring_ids, "zernike", normalization_mode=normalization_mode)
        plot_limits[(mode_name, "physical")] = _compute_panel_limits(physical_data, ring_ids, summary_types)
        plot_limits[(mode_name, "zernike")] = _compute_panel_limits(zernike_data, ring_ids, summary_types)

    return ring_ids, summary_types, plot_limits



def _feature_mask(df, selections):
    mask = pd.Series(True, index=df.index)
    for feature, (lower, upper) in selections.items():
        mask &= df[feature].between(lower, upper, inclusive="both")
    return mask



def load_dataframe(_=None):
    with status:
        status.clear_output()
        try:
            path = Path(csv_path.value).expanduser()
            if not path.exists():
                print(f"File non trovato: {path}")
                STATE["df"] = None
                return

            df = pd.read_csv(path)
            missing = [feature for feature in FEATURES if feature not in df.columns]
            if missing:
                print(f"Colonne mancanti: {', '.join(missing)}")
                STATE["df"] = None
                return

            for column in df.columns:
                if column not in {"complex_name", "protein1_file", "protein2_file", "gyration_radius_note", "summary_type"}:
                    df[column] = pd.to_numeric(df[column], errors="coerce")

            STATE["df"] = df
            STATE["summary_path"] = path
            STATE["ring_ids"], STATE["summary_types"], STATE["plot_limits"] = _prepare_state(df)

            for feature, slider in sliders.items():
                _configure_slider(slider, df[feature])

            print(f"Caricato: {path}")
            print(f"Righe: {len(df)}")
            print(f"Ring disponibili: {len(STATE['ring_ids'])}")
            print(f"Summary types: {', '.join(STATE['summary_types'])}")
        except Exception as exc:
            STATE["df"] = None
            print(f"Errore nel caricamento: {exc}")


load_button.on_click(load_dataframe)



def update_plot(gyration_radius, flatness, roughness, scalar_prod):
    df = STATE["df"]
    if df is None:
        print("Carica prima il CSV e premi 'Carica DataFrame'.")
        return

    ring_ids = STATE["ring_ids"]
    summary_types = STATE["summary_types"]
    if not ring_ids or not summary_types:
        print("Nessun dato RDF disponibile nel CSV caricato.")
        return

    selections = {
        "gyration_radius": gyration_radius,
        "flatness": flatness,
        "roughness": roughness,
        "scalar_prod": scalar_prod,
    }
    filtered = df.loc[_feature_mask(df, selections)].copy()
    if filtered.empty:
        print("Nessun dato")
        return

    colors = {
        "weighted": "#1f77b4",
        "normal": "#ff7f0e",
    }
    markers = {
        "weighted": "o",
        "normal": "s",
    }

    panel_data = {}
    for mode_name, normalization_mode, mode_label in PLOT_MODES:
        physical_data = _aggregate_metric_by_summary_type(filtered, summary_types, ring_ids, "physical", normalization_mode=normalization_mode)
        zernike_data = _aggregate_metric_by_summary_type(filtered, summary_types, ring_ids, "zernike", normalization_mode=normalization_mode)
        panel_data[mode_name] = {
            "label": mode_label,
            "physical": physical_data,
            "zernike": zernike_data,
        }

    fig, axes = plt.subplots(len(PLOT_MODES), 2, figsize=(16, 18), sharex=True)
    for row_index, (mode_name, _, mode_label) in enumerate(PLOT_MODES):
        show_legend = (row_index == 0) and ("weighted" in summary_types and "normal" in summary_types)
        _plot_metric_panel(
            axes[row_index, 0],
            panel_data[mode_name]["physical"],
            ring_ids,
            summary_types,
            colors,
            markers,
            f"{mode_label} - Physical Distance per Ring",
            "Physical Distance (Å)",
            STATE["plot_limits"][(mode_name, "physical")],
            show_legend=show_legend,
        )
        _plot_metric_panel(
            axes[row_index, 1],
            panel_data[mode_name]["zernike"],
            ring_ids,
            summary_types,
            colors,
            markers,
            f"{mode_label} - Zernike Distance per Ring",
            "Zernike Distance",
            STATE["plot_limits"][(mode_name, "zernike")],
            show_legend=False,
        )

        axes[row_index, 0].text(
            -0.18,
            0.5,
            mode_label,
            transform=axes[row_index, 0].transAxes,
            rotation=90,
            va="center",
            ha="center",
            fontsize=12,
            fontweight="bold",
        )

    fig.suptitle(
        f"Radial Functions with Uncertainties ({STATE['summary_path'].stem}) - filtered rows: {len(filtered)}",
        fontsize=14,
        fontweight="bold",
        y=0.995,
    )
    fig.tight_layout()
    plt.show()
    plt.close(fig)


plot_output = widgets.interactive_output(
    update_plot,
    {
        "gyration_radius": sliders["gyration_radius"],
        "flatness": sliders["flatness"],
        "roughness": sliders["roughness"],
        "scalar_prod": sliders["scalar_prod"],
    },
)

controls = widgets.VBox(
    [
        widgets.HBox([csv_path, load_button]),
        widgets.VBox([
            sliders["gyration_radius"],
            sliders["flatness"],
            sliders["roughness"],
            sliders["scalar_prod"],
        ]),
        status,
        plot_output,
    ]
)

display(controls)

In [5]:
from io import BytesIO
from pathlib import Path

from PIL import Image

GIF_OUTPUT_DIR = Path("output_files/default/gifs")
GIF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _figure_to_image(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def _render_normalized_panel(df_subset, ring_ids, summary_types, feature_name):
    colors = {
        "weighted": "#1f77b4",
        "normal": "#ff7f0e",
    }
    markers = {
        "weighted": "o",
        "normal": "s",
    }
    metric_data_physical = _aggregate_metric_by_summary_type(
        df_subset,
        summary_types,
        ring_ids,
        "physical",
        normalization_mode="minmax",
    )
    metric_data_zernike = _aggregate_metric_by_summary_type(
        df_subset,
        summary_types,
        ring_ids,
        "zernike",
        normalization_mode="minmax",
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    show_legend = "weighted" in summary_types and "normal" in summary_types

    _plot_metric_panel(
        axes[0],
        metric_data_physical,
        ring_ids,
        summary_types,
        colors,
        markers,
        "Min-max normalized RDF - Physical Distance per Ring",
        "Normalized value",
        (0.0, 1.0),
        show_legend=show_legend,
    )
    _plot_metric_panel(
        axes[1],
        metric_data_zernike,
        ring_ids,
        summary_types,
        colors,
        markers,
        "Min-max normalized RDF - Zernike Distance per Ring",
        "Normalized value",
        (0.0, 1.0),
        show_legend=False,
    )

    fig.suptitle(f"{feature_name} lower bound animation - {len(df_subset)} rows", fontsize=14, fontweight="bold")
    fig.tight_layout()
    return _figure_to_image(fig)


def make_slider_gif(feature_name, n_frames=120, output_dir=GIF_OUTPUT_DIR):
    df = STATE["df"]
    if df is None:
        raise RuntimeError("Carica prima il CSV con il pulsante 'Carica DataFrame'.")

    if not STATE["ring_ids"] or not STATE["summary_types"]:
        STATE["ring_ids"], STATE["summary_types"], STATE["plot_limits"] = _prepare_state(df)

    feature_values = _finite_series(df[feature_name])
    if feature_values.empty:
        raise ValueError(f"Nessun valore valido trovato per {feature_name}")

    lower_min = float(feature_values.min())
    lower_max = float(feature_values.max())
    if np.isclose(lower_min, lower_max):
        lower_max = lower_min + 1.0

    lower_bounds = np.linspace(lower_min, lower_max, n_frames)
    upper_fixed = lower_max
    ring_ids = STATE["ring_ids"]
    summary_types = STATE["summary_types"]

    frames = []
    for idx, lower_bound in enumerate(lower_bounds, start=1):
        selections = {
            feature_name: (float(lower_bound), float(upper_fixed)),
        }
        for other_feature in FEATURES:
            if other_feature != feature_name:
                other_values = _finite_series(df[other_feature])
                if other_values.empty:
                    continue
                selections[other_feature] = (float(other_values.min()), float(other_values.max()))

        filtered = df.loc[_feature_mask(df, selections)].copy()
        if filtered.empty:
            fig, ax = plt.subplots(figsize=(12, 4))
            ax.axis("off")
            ax.text(0.5, 0.5, f"Nessun dato\n{feature_name} >= {lower_bound:.4f}", ha="center", va="center", fontsize=16)
            frames.append(_figure_to_image(fig))
            continue

        frames.append(_render_normalized_panel(filtered, ring_ids, summary_types, f"{feature_name} >= {lower_bound:.4f}"))

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    gif_path = output_dir / f"minmax_norm_{feature_name}.gif"
    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=90,
        loop=0,
        optimize=False,
    )
    print(f"GIF salvata: {gif_path}")
    return gif_path


def make_all_gifs(n_frames=120, output_dir=GIF_OUTPUT_DIR):
    gif_paths = []
    for feature_name in FEATURES:
        gif_paths.append(make_slider_gif(feature_name, n_frames=n_frames, output_dir=output_dir))
    return gif_paths


# Esegui questa cella dopo aver caricato il CSV con la cella interattiva.
# Se vuoi una preview più rapida, abbassa n_frames a 60.
if STATE["df"] is None:
    print("Carica prima il CSV nella cella interattiva e poi riesegui questa cella.")
else:
    make_all_gifs(n_frames=120)


GIF salvata: output_files/default/gifs/minmax_norm_gyration_radius.gif
GIF salvata: output_files/default/gifs/minmax_norm_flatness.gif
GIF salvata: output_files/default/gifs/minmax_norm_roughness.gif
GIF salvata: output_files/default/gifs/minmax_norm_scalar_prod.gif
